# Pre-Game Features

The cleaned data from `data_exploration.ipynb` holds box-score stats from **the game itself** (yards, turnovers, third downs, ...). Those numbers aren't known until the game is over, so using them to predict the winner would be cheating.

This notebook turns them into numbers that **are** known before kickoff: each team's averages from its earlier games. It saves `datasets/cfb_pregame_features.csv`, which `modeling.ipynb` uses.

**Run `data_exploration.ipynb` first** (it writes `datasets/cfb_fbs_clean.csv`).

Steps:
1. Split each game into two rows, one for each team.
2. For each team and season, average every stat over the team's **earlier** games.
3. For a team's first few games of a season, use its averages from **last season** instead.
4. Put the home and away numbers back on one row per game and take **home minus away**.

## Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
# Until a team has played this many games in a season, use last season's averages instead.
# (On the 2024 season, using 1, 2, 3 or 4 games here all gave about the same accuracy.)
MIN_GAMES = 3

# The stats we average for each team. Each one is built in make_team_rows() below.
STATS = [
    'points_scored',
    'points_allowed',
    'yards_gained',
    'yards_allowed',
    'turnovers_lost',
    'turnovers_forced',
    'third_down_pct',
    'penalty_yards',
    'win_pct',
]

## Load the Cleaned Games

In [ ]:
games = pd.read_csv('datasets/cfb_fbs_clean.csv', parse_dates=['date'])

# Put the games in the order they were played and give each one an id
games = games.sort_values('date').reset_index(drop=True)
games['game_id'] = games.index

print(len(games), 'games')

## Step 1: One Row per Team per Game

Each game has a home team and an away team. To track a team over its season, we want one row per team per game, no matter which side it was on. `make_team_rows()` builds those rows for one side.

In [ ]:
def make_team_rows(games, side):
    """
    Build one row per game for the team on `side` ('home' or 'away').
    Each row has what that team did in the game and what it gave up.
    """
    if side == 'home':
        opp = 'away'
    else:
        opp = 'home'

    rows = pd.DataFrame({
        'game_id': games['game_id'],
        'season': games['season'],
        'date': games['date'],
        'team': games[side],
        'points_scored': games['score_' + side],
        'points_allowed': games['score_' + opp],
        'yards_gained': games['total_yards_' + side],
        'yards_allowed': games['total_yards_' + opp],
        # fum_ and int_ count the fumbles lost and interceptions thrown by that team
        'turnovers_lost': games['fum_' + side] + games['int_' + side],
        'turnovers_forced': games['fum_' + opp] + games['int_' + opp],
        'third_down_pct': games['third_down_comp_' + side] / games['third_down_att_' + side],
        'penalty_yards': games['pen_yards_' + side],
        # 1 if this team won, 0 if not. Averaged over games, this is the team's win %.
        'win_pct': (games['score_' + side] > games['score_' + opp]).astype(int),
    })

    # A few games list 0 third-down attempts, which makes the percentage blank or infinite.
    # Treat those as missing.
    rows['third_down_pct'] = rows['third_down_pct'].replace([np.inf, -np.inf], np.nan)
    return rows

In [ ]:
home_rows = make_team_rows(games, 'home')
away_rows = make_team_rows(games, 'away')

team_games = pd.concat([home_rows, away_rows])
team_games = team_games.sort_values(['team', 'date']).reset_index(drop=True)

print(len(team_games), 'team-game rows (2 per game)')
team_games[team_games['season'] == 2025].head()

## Step 2: Average of Earlier Games This Season

For every game, we want the team's average **before** that game. `shift(1)` moves each value down one row, so a game never sees its own stats. `expanding().mean()` then averages everything above it. We do this separately for every team and season.

In [ ]:
def average_of_earlier_games(values):
    """
    `values` is one stat for one team in one season, in the order the games were played.
    Returns, for each game, the average of that stat over the team's EARLIER games.
    The first game of the season has no earlier games, so it gets NaN.
    """
    return values.shift(1).expanding().mean()


for stat in STATS:
    by_team_season = team_games.groupby(['team', 'season'])[stat]
    team_games['season_avg_' + stat] = by_team_season.transform(average_of_earlier_games)

# How many games the team had already played this season before this game
team_games['games_played'] = team_games.groupby(['team', 'season']).cumcount()

Quick check on one team: the `season_avg_` columns should only reflect the games **above** each row. Tennessee's first game has nothing before it, so it's blank.

In [ ]:
check_cols = ['date', 'team', 'games_played', 'points_scored', 'season_avg_points_scored',
              'win_pct', 'season_avg_win_pct']
is_tennessee_2025 = (team_games['team'] == 'Tennessee') & (team_games['season'] == 2025)
team_games.loc[is_tennessee_2025, check_cols].head(6)

## Step 3: Use Last Season for the First Few Games

An average over 0-2 games isn't worth much, so for a team's first `MIN_GAMES` games we use its average from the whole previous season instead. Last season's win % is also kept as its own feature for every game.

Teams new to FBS (and every team in 2002, the first season in the data) have no previous season, so they keep this season's numbers.

In [ ]:
def last_season_averages(team_games):
    """
    Each team's average for every stat over its whole previous season.
    We average each team's season, then add 1 to the season number so that
    a team's 2024 averages line up with its 2025 games.
    """
    season_avgs = team_games.groupby(['team', 'season'])[STATS].mean().reset_index()
    season_avgs['season'] = season_avgs['season'] + 1

    new_names = {}
    for stat in STATS:
        new_names[stat] = 'last_season_' + stat
    return season_avgs.rename(columns=new_names)


team_games = team_games.merge(last_season_averages(team_games), on=['team', 'season'], how='left')

In [ ]:
for stat in STATS:
    early_in_season = team_games['games_played'] < MIN_GAMES
    has_last_season = team_games['last_season_' + stat].notna()
    use_last_season = early_in_season & has_last_season

    # Pre-game value: last season's average early on, this season's average after that
    team_games['pre_' + stat] = np.where(use_last_season,
                                         team_games['last_season_' + stat],
                                         team_games['season_avg_' + stat])

# Last season's win % as its own feature (blank for teams with no previous FBS season)
team_games['pre_last_season_win_pct'] = team_games['last_season_win_pct']

# These are the stats that become features
FEATURE_STATS = STATS + ['last_season_win_pct']

## Step 4: Back to One Row per Game

Now we attach the home team's pre-game numbers and the away team's pre-game numbers to each game, then take home minus away. A positive `diff_points_scored` means the home team has been scoring more.

In [ ]:
def add_team_features(games, team_games, side):
    """
    Attach the pre-game numbers of the team on `side` ('home' or 'away') to each game.
    The new columns are named like home_points_scored or away_points_scored.
    """
    pre_cols = []
    new_names = {'team': side}
    for stat in FEATURE_STATS:
        pre_cols.append('pre_' + stat)
        new_names['pre_' + stat] = side + '_' + stat

    side_features = team_games[['game_id', 'team'] + pre_cols].rename(columns=new_names)
    return games.merge(side_features, on=['game_id', side], how='left')


games = add_team_features(games, team_games, 'home')
games = add_team_features(games, team_games, 'away')

for stat in FEATURE_STATS:
    games['diff_' + stat] = games['home_' + stat] - games['away_' + stat]

## Step 5: Keep Only What's Known Before Kickoff and Save

None of the same-game box-score columns or scores are kept. `home_win` stays as the target.

In [ ]:
info_cols = ['season', 'week', 'date', 'postseason', 'home', 'away', 'neutral', 'conf_game',
             'home_is_ranked', 'away_is_ranked', 'rank_home_filled', 'rank_away_filled']

home_cols = []
away_cols = []
diff_cols = []
for stat in FEATURE_STATS:
    home_cols.append('home_' + stat)
    away_cols.append('away_' + stat)
    diff_cols.append('diff_' + stat)

pregame = games[info_cols + home_cols + away_cols + diff_cols + ['home_win']]
pregame.to_csv('datasets/cfb_pregame_features.csv', index=False)

print(pregame.shape)
pregame.head()

### How Much Is Missing?

Blanks are left for the few games where a team has no earlier games this season **and** no previous season (mostly teams in their first FBS season, plus all of 2002). `modeling.ipynb` fills these with the median from the training data.

In [ ]:
from_2004 = pregame[pregame['season'] >= 2004]
(from_2004[diff_cols].isna().mean() * 100).round(2).rename('% missing (2004 on)')

## Do the Pre-Game Features Line Up With Winning?

Same idea as the correlation chart in `data_exploration.ipynb`, but with the pre-game numbers. These correlations are weaker than the same-game ones, which is expected: knowing how a team has played so far tells you less than knowing how it played in the game itself.

In [ ]:
correlations = from_2004[diff_cols].corrwith(from_2004['home_win']).sort_values()

correlations.plot.barh(figsize=(8, 5))
plt.title('Correlation of pre-game home-minus-away stats with home win (2004-2025)')
plt.xlabel('Correlation with home win')
plt.show()